
### SCD Type 2 on Card Table


In [0]:
from pyspark.sql.functions import *

#  Source
df = spark.read.format("delta").table("bankingdata.silver.card")

In [0]:
df.columns

### Hash Column at Source


In [0]:
df_hash = df.withColumn(
    "src_hash",
    crc32(concat_ws("||",
        col("CardID").cast("string"),
        col("AccountID").cast("string"),
        col("CardType"),
        col("CardLimit").cast("string"),
        col("ExpiryDate").cast("string")
    ))
)


### Target Table


In [0]:
%sql
CREATE TABLE IF NOT EXISTS bankingdata.gold.dim_card
(
    CardID INT,
    AccountID INT,
    CardType STRING,
    CardLimit DOUBLE,
    ExpiryDate DATE,
    ModifiedDate TIMESTAMP,

    HASHVALUE STRING,

    CREATEDDATE TIMESTAMP,
    CREATEDBY STRING,
    UPDATEDDATE TIMESTAMP,
    UPDATEDBY STRING,

    IS_ACTIVE INT
)

In [0]:
# Target table
from delta.tables import DeltaTable
table_name = "bankingdata.gold.dim_card"
df_tgt = DeltaTable.forName(spark, table_name)

### Detect New or Changed Records

In [0]:
df_new = (
    df_hash.alias("src")
    .join(
        df_tgt.toDF().alias("tgt"),
        (col("src.CardID") == col("tgt.CardID")) &
        (col("src.src_hash") == col("tgt.HASHVALUE")),
        "left_anti"
    )
)

### Identify Changed Records

In [0]:
LatestRecord = (
    df_new.alias("new")
    .join(
        df_tgt.toDF().alias("old"),
        col("new.CardID") == col("old.CardID"),
        "inner"
    )
    .where(
        (col("new.src_hash") != col("old.HASHVALUE")) &
        (col("old.IS_ACTIVE") == 1)
    )
    .select(
        col("new.CardID").alias("MergeKey"),
        col("new.*")
    )
)

### Prepare New Records

In [0]:
from pyspark.sql.functions import lit

LatestRecord1 = (
    df_new
    .select(
        lit(None).alias("MergeKey"),
        "*"
    )
)

### Combine Changed and New Records

In [0]:
updates = LatestRecord.union(LatestRecord1)

### Apply SCD Type-2 Using Delta MERGE

In [0]:
(
    df_tgt.alias("tgt")
    .merge(
        updates.alias("src"),
        "tgt.CardID = src.MergeKey AND tgt.IS_ACTIVE = 1"
    )
    
    # Expire old record if changed
    .whenMatchedUpdate(
        condition="tgt.IS_ACTIVE = 1 AND tgt.HASHVALUE != src.src_hash",
        set={
            "UPDATEDDATE": current_timestamp(),
            "UPDATEDBY": lit("Databricks-Updated"),
            "IS_ACTIVE": lit(0)
        }
    )
    
    # Insert new record
    .whenNotMatchedInsert(
        values={
            "CardID": "src.CardID",
            "AccountID": "src.AccountID",
            "CardType": "src.CardType",
            "CardLimit": "src.CardLimit",
            "ExpiryDate": "src.ExpiryDate",
            "ModifiedDate": "src.ModifiedDate",
            "HASHVALUE": "src.src_hash",
            "CREATEDDATE": current_timestamp(),
            "CREATEDBY": lit("databricks"),
            "UPDATEDDATE": to_timestamp(lit("9999-01-01 00:00:00")),
            "UPDATEDBY": lit("databricks"),
            "IS_ACTIVE": lit(1)
        }
    )
    
    .execute()
)